In [10]:
#Buat Data simulasi
import pandas as pd

hyper = pd.DataFrame({
    "tanggal": ["2026-06-20"],
    "amount": [1000],
    "customer": ["John"]
})

express = pd.DataFrame({
    "trx_date": ["2026-06-20"],
    "sales_value": [1200],
    "customer_name": ["Mary"]
})

online = pd.DataFrame({
    "order_date": ["2026-06-20"],
    "total_amount": [900],
    "customer_id": ["Alex"]
})

wholesale = pd.DataFrame({
    "invoice_date": ["2026-06-20"],
    "invoice_amount": [3000],
    "client": ["ABC Corp"]
})

fresh = pd.DataFrame({
    "delivery_date": ["2026-06-20"],
    "delivery_value": [1500],
    "buyer": ["Fresh Mart"]
})

In [11]:
canonical_schema = {
    "transaction_date": "datetime64[ns]",
    "sales_amount": "float64",
    "customer": "object"
}

schema_registry = {

    "hyper": {
        "tanggal": "transaction_date",
        "amount": "sales_amount",
        "customer": "customer"
    },

    "express": {
        "trx_date": "transaction_date",
        "sales_value": "sales_amount",
        "customer_name": "customer"
    },

    "online": {
        "order_date": "transaction_date",
        "total_amount": "sales_amount",
        "customer_id": "customer"
    },

    "wholesale": {
        "invoice_date": "transaction_date",
        "invoice_amount": "sales_amount",
        "client": "customer"
    },

    "fresh": {
        "delivery_date": "transaction_date",
        "delivery_value": "sales_amount",
        "buyer": "customer"
    }

}

In [12]:

hyper_sp = spark.createDataFrame(hyper)
express_sp = spark.createDataFrame(express)
online_sp = spark.createDataFrame(online)
wholesale_sp = spark.createDataFrame(wholesale)
fresh_sp = spark.createDataFrame(fresh)

Konfigurasi hadoop

In [ ]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] += ";" + r"C:\hadoop\bin"

initialisasi spark

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("MegaMart")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config(
        "spark.hadoop.hadoop.home.dir",
        r"C:\hadoop"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()


spark.sparkContext._jsc.hadoopConfiguration().set(
    "hadoop.home.dir",
    r"C:\hadoop"
)

Harmonisasi skema

In [13]:
from pyspark.sql.functions import *

def harmonize(df, source_name):

    mapping = schema_registry[source_name]

    for src, tgt in mapping.items():

        if src in df.columns:
            df = df.withColumnRenamed(src, tgt)

    required_cols = [
        "transaction_date",
        "sales_amount",
        "customer"
    ]

    for col_name in required_cols:

        if col_name not in df.columns:
            df = df.withColumn(
                col_name,
                lit(None)
            )

    return (
        df
        .select(*required_cols)
        .withColumn(
            "transaction_date",
            to_date("transaction_date")
        )
        .withColumn(
            "sales_amount",
            col("sales_amount")
            .cast("double")
        )
        .withColumn(
            "source_system",
            lit(source_name)
        )
        .withColumn(
            "load_timestamp",
            current_timestamp()
        )
    )

In [14]:
hyper_h = harmonize(hyper_sp, "hyper")
express_h = harmonize(express_sp, "express")
online_h = harmonize(online_sp, "online")
wholesale_h = harmonize(wholesale_sp, "wholesale")
fresh_h = harmonize(fresh_sp, "fresh")

Unified 5 silo tabel

In [15]:
unified = (
    hyper_h
    .unionByName(express_h)
    .unionByName(online_h)
    .unionByName(wholesale_h)
    .unionByName(fresh_h)
)

unified.show()

+----------------+------------+----------+-------------+--------------------+
|transaction_date|sales_amount|  customer|source_system|      load_timestamp|
+----------------+------------+----------+-------------+--------------------+
|      2026-06-20|      1000.0|      John|        hyper|2026-06-21 10:00:...|
|      2026-06-20|      1200.0|      Mary|      express|2026-06-21 10:00:...|
|      2026-06-20|       900.0|      Alex|       online|2026-06-21 10:00:...|
|      2026-06-20|      3000.0|  ABC Corp|    wholesale|2026-06-21 10:00:...|
|      2026-06-20|      1500.0|Fresh Mart|        fresh|2026-06-21 10:00:...|
+----------------+------------+----------+-------------+--------------------+



Write to delta

In [16]:
unified = unified.repartition(
    8,
    "transaction_date"
)

(
    unified
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy(
        "transaction_date"
    )
    .save(
        "../data/silver/unified_transactions"
    )
)

Load kembali

In [ ]:
silver = (
    spark.read
    .format("delta")
    .load(
        "../data/silver/unified_transactions"
    )
)

silver.show()

+----------------+------------+----------+-------------+--------------------+
|transaction_date|sales_amount|  customer|source_system|      load_timestamp|
+----------------+------------+----------+-------------+--------------------+
|      2026-06-20|      1000.0|      John|        hyper|2026-06-21 10:01:...|
|      2026-06-20|      1200.0|      Mary|      express|2026-06-21 10:01:...|
|      2026-06-20|       900.0|      Alex|       online|2026-06-21 10:01:...|
|      2026-06-20|      3000.0|  ABC Corp|    wholesale|2026-06-21 10:01:...|
|      2026-06-20|      1500.0|Fresh Mart|        fresh|2026-06-21 10:01:...|
+----------------+------------+----------+-------------+--------------------+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 53959)
Traceback (most recent call last):
  File "C:\Python311\Lib\socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "C:\Python311\Lib\socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "C:\Python311\Lib\socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "C:\Python311\Lib\socketserver.py", line 755, in __init__
    self.handle()
  File "d:\Projectku\BigData\MegaMart\.venv\Lib\site-packages\pyspark\accumulators.py", line 295, in handle
    poll(accum_updates)
  File "d:\Projectku\BigData\MegaMart\.venv\Lib\site-packages\pyspark\accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "d:\Projectku\BigData\MegaMart\.venv\Lib\site-packages\pyspark\accumula